# ATTS-RAG: GPU runs, part 2 (free Kaggle GPU)

| Job | Feedback point | What it produces |
|---|---|---|
| BIPIA | 1, 3 | Microsoft's indirect prompt-injection benchmark (150 cases, 5 defences) through the whole pipeline, scored with BIPIA's own rules plus the qwen2.5:14b judge for the attacks BIPIA scores with GPT |
| Poisoning with other generators | 3, 5 | The 64 held-out poisoning cases (11 defences) with Llama 3, Gemma 4, Mistral 7B and Phi-4 14B writing the answer. The defences' LLM judge stays Llama 3, so only the generator changes |
| Second independent judge | 5 | Gemma 3 27B labels RAGTruth sentences (against the human labels) and every sentence of our four generators (against the first judge) |

**Settings** (right-hand panel → *Session options*): **Accelerator** *GPU T4 x2*, **Internet** *On*.

**To run:** *Save Version* → *Save & Run All (Commit)*. Takes about 4–6 hours. When done: the version's **Output** tab → download `atts_gpu_results_2.zip`.

Jobs that would start after `TIME_LIMIT_H` are skipped, and the zip is rewritten after every job.

In [ ]:
# ---- settings ------------------------------------------------------------------------------------
REPO = "https://github.com/SakthiQ/ATTS-RAG.git"
BRANCH = "claude/layer1-detection"
POISON_GENERATORS = ["llama3", "gemma4:e4b", "mistral:7b", "phi4:14b"]
GUARD_MODEL = "llama3"          # the defences' LLM judge, held fixed across generators
JUDGE1 = "qwen2.5:14b"          # scores the BIPIA attacks that BIPIA itself scores with GPT
JUDGE2 = "gemma3:27b"           # second independent judge
JOBS = {"bipia": True, "poison_generators": True, "judge2": True}
TIME_LIMIT_H = 11.0
SMOKE = False                   # True: a few cases per job, to check the setup

In [ ]:
# ---- setup: code, packages, benchmarks, Ollama -------------------------------------------------------
import json, os, shutil, subprocess, sys, time, urllib.request

T0 = time.time()
BASE = "/tmp/atts"
REPO_DIR = f"{BASE}/ATTS-RAG"
RT_DATA = f"{BASE}/ragtruth/dataset"
BIPIA_DIR = f"{BASE}/bipia"
CASES = f"{BASE}/bipia_cases"
OUT = "/kaggle/working/atts_gpu2" if os.path.isdir("/kaggle/working") else f"{BASE}/out2"
R = f"{OUT}/results"
HOST = "http://127.0.0.1:11434"
for d in (BASE, OUT, R, f"{OUT}/logs"):
    os.makedirs(d, exist_ok=True)
STATUS = {}


def say(*a):
    print(f"[{(time.time() - T0) / 3600:4.1f} h]", *a, flush=True)


def sh(cmd, log=None, cwd=None, env=None):
    out = open(log, "a") if log else None
    rc = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=out, stderr=subprocess.STDOUT if out else None).returncode
    if out:
        out.close()
    return rc


def left_h():
    return TIME_LIMIT_H - (time.time() - T0) / 3600


sh("nvidia-smi -L")
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
assert NGPU > 0, "No GPU: set Accelerator to GPU in the notebook settings"
if not os.path.isdir(REPO_DIR):
    assert sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR}") == 0, "git clone failed: is Internet on?"
COMMIT = subprocess.run(f"git -C {REPO_DIR} rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
say("ATTS-RAG commit", COMMIT, "|", NGPU, "GPU(s)")
assert os.path.isdir(f"{REPO_DIR}/eval/judge_inputs"), "this branch has no eval/judge_inputs: push the latest commit first"
if not os.path.isdir(RT_DATA):
    sh(f"git clone -q --depth 1 https://github.com/ParticleMedia/RAGTruth.git {BASE}/ragtruth")
if not os.path.isdir(BIPIA_DIR):
    sh(f"git clone -q --depth 1 https://github.com/microsoft/BIPIA.git {BIPIA_DIR}")

PKGS = ("langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters langchain-ollama "
        "chromadb tiktoken rank_bm25 loguru tenacity pypdf pdfplumber pymupdf python-docx rapidocr-onnxruntime "
        "python-dotenv emoji langdetect rapidfuzz")
assert sh(f"{sys.executable} -m pip install -q -U {PKGS}", log=f"{OUT}/logs/pip.log") == 0, "pip install failed, see logs/pip.log"

if shutil.which("ollama") is None:
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", log=f"{OUT}/logs/apt.log")
    assert sh("curl -fsSL https://ollama.com/install.sh | sh", log=f"{OUT}/logs/ollama_install.log") == 0, "Ollama install failed"
OLLAMA_ENV = dict(os.environ, OLLAMA_MODELS=f"{BASE}/ollama", OLLAMA_KEEP_ALIVE="30m", OLLAMA_HOST="127.0.0.1:11434")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=OLLAMA_ENV, stdout=open(f"{OUT}/logs/ollama_serve.log", "a"),
                               stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(HOST + "/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
say("Ollama up")

ENV = dict(os.environ, PYTHONPATH=".", HF_HOME=f"{BASE}/hf", TOKENIZERS_PARALLELISM="false",
           JUDGE_OLLAMA_HOST=HOST, OLLAMA_HOST="127.0.0.1:11434")
LIM = "--limit 3" if SMOKE else ""
BIPIA_CONDITIONS = ("none (no defense)|prompt-only (documents are untrusted data)|LLM-judge chunk filter|"
                    "guard + L2 + L3 (full)|deep guard + L2 + L3 (full, deep scan)")


def save():
    json.dump({"commit": COMMIT, "smoke": SMOKE, "gpus": NGPU, "hours": round((time.time() - T0) / 3600, 2), "jobs": STATUS},
              open(f"{OUT}/status.json", "w"), indent=1)
    if os.path.isdir("/kaggle/working"):
        shutil.make_archive("/kaggle/working/atts_gpu_results_2", "zip", OUT)


def job(name, cmd, env=None, need_h=0.3):
    if left_h() < need_h:
        STATUS[name] = "skipped (time limit)"
        say("SKIP", name, "(time limit)")
        return False
    say("START", name)
    t = time.time()
    log = f"{OUT}/logs/{name}.log"
    rc = sh(cmd, log=log, cwd=REPO_DIR, env=dict(ENV, **(env or {})))
    STATUS[name] = f"{'ok' if rc == 0 else 'FAILED (exit ' + str(rc) + ')'} in {(time.time() - t) / 60:.0f} min"
    say(STATUS[name], name)
    if rc:
        print("".join(open(log, errors="replace").readlines()[-15:]))
    save()
    return rc == 0


def pull(model):
    say("pull", model)
    return sh(f"ollama pull {model}", log=f"{OUT}/logs/pull.log", env=OLLAMA_ENV) == 0


def drop(model):
    sh(f"ollama rm {model}", log=f"{OUT}/logs/pull.log", env=OLLAMA_ENV)


def tag(model):
    return model.replace(":", "-").replace(".", "")


save()

In [ ]:
# ---- 1. BIPIA through the whole pipeline, then scoring ----------------------------------------------------
# The cases are rebuilt from the public BIPIA repository with the same seed (identical to the author's cases.json).
if JOBS["bipia"]:
    pull("llama3")
    pull(JUDGE1)
    if job("bipia_build", f"python -u scripts/bipia_cases.py build --bipia {BIPIA_DIR} --out {CASES}", need_h=0.1):
        if job("bipia_run", f"python -u scripts/evaluate_pipeline.py --suite poison-large --poison-dir {CASES} "
               f"--extra-corpus eval/corpus_heldout --conditions \"{BIPIA_CONDITIONS}\" --cache {OUT}/cache_bipia.jsonl "
               f"{LIM} --out {R}/bipia", need_h=1.0):
            job("bipia_score", f"python -u scripts/bipia_cases.py score --bipia {BIPIA_DIR} --cases {CASES} --results {R}/bipia "
                f"--host {HOST} --judge-model {JUDGE1}", need_h=0.3)
    drop(JUDGE1)

In [ ]:
# ---- 2. The 64 held-out poisoning cases with each generator (defences' judge fixed to Llama 3) -----------------
if JOBS["poison_generators"]:
    pull(GUARD_MODEL)
    for m in POISON_GENERATORS:
        t = tag(m)
        if left_h() < (1.5 if "14b" in m else 1.0):
            STATUS[f"poison_v2_{t}"] = "skipped (time limit)"
            continue
        if m != GUARD_MODEL and not pull(m):
            STATUS[f"poison_v2_{t}"] = "FAILED (ollama pull)"
            continue
        job(f"poison_v2_{t}", f"python -u scripts/evaluate_pipeline.py --suite poison-large --poison-dir poison_large_v2 "
            f"--extra-corpus eval/corpus_heldout --model {m} --guard-model {GUARD_MODEL} --cache {OUT}/cache_poison.jsonl "
            f"{LIM} --out {R}/poison_v2_{t}", need_h=0.8)
        if m != GUARD_MODEL:
            drop(m)
    drop(GUARD_MODEL)

In [ ]:
# ---- 3. Second independent judge: RAGTruth (vs human labels) and our four generators (vs the first judge) -------
if JOBS["judge2"]:
    pull(JUDGE2)
    n_rt = 10 if SMOKE else 300
    job("judge2_ragtruth", f"python -u scripts/judge_ragtruth.py --n {n_rt} --data {RT_DATA} "
        f"--items eval/judge_inputs/ragtruth_items.jsonl --judge-model {JUDGE2} --host {HOST} --out {R}/judge2_ragtruth", need_h=0.5)
    for gen in ("llama3", "gemma4-e4b", "mistral-7b", "phi4-14b"):
        d = f"{R}/judge2_{gen}"
        os.makedirs(d, exist_ok=True)
        lines = open(f"{REPO_DIR}/eval/judge_inputs/claims_{gen}.jsonl", encoding="utf-8").readlines()
        open(f"{d}/claims.jsonl", "w", encoding="utf-8").writelines(lines[:5] if SMOKE else lines)
        if job(f"judge2_{gen}", f"python -u scripts/independent_judge.py judge --out {d} --host {HOST} --judge-model {JUDGE2}", need_h=0.4):
            job(f"judge2_report_{gen}", f"python -u scripts/independent_judge.py report --out {d}", need_h=0.05)

In [ ]:
# ---- done: status and the head of each report ----------------------------------------------------------------
import glob
save()
say("DONE")
for k, v in STATUS.items():
    print(f"  {k:30s} {v}")
for rep in sorted(glob.glob(f"{R}/*/*report*.md")):
    print("=" * 100, "\n", rep.replace(R + "/", ""), "\n")
    print("".join(open(rep, errors="replace").readlines()[:25]))